# SLC46A1 engraftment, step 7: the admissible construct scored as a set, and pairs at chicken 322

Part A scores the construct that leaves the pharmacophore set untouched. Its
four rescues were each chosen in the background its own scan used, three against
wild-type chicken and one against the construct, so the set has never been
scored together, and step 5 showed that scoring together is where rescues are
lost. Every primary is rescored with the other nineteen and all six rescuer
positions in place. Two columns are scored alongside it for scale, the
wild-type background against step 1 and the primaries-only background against
step 5, and the run stops if either disagrees.

Part B is the pair scan at chicken 322, the one high-risk substitution with
nothing crossing in any scan so far. Its candidate set is the union of the ten
positions most strongly coupled to it and the ten nearest it in the deposited
coordinates, both read from the files that computed them. The scan runs in the
construct background, not against wild-type chicken, since that is the
background the finished sequence has, and the fifteen singles are rescored there
too so synergy is measured on one scale.

It stops on fatal conditions instead of warning, checkpoints to Drive after
every batch, and resumes from the checkpoint if the runtime is lost. Part A is
minutes and Part B is about two and a half hours on a T4. Run every cell in
order.


In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'esm2_step7')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/admissible_construct.json',
            'data/coupling_guided_rescue.json',
            'data/structural_candidates.json',
            'data/construct_report.json',
            'data/pharmacophore_position_set.tsv',
            'data/primary_substitution_risk.tsv',
            'data/esm2_step2/rescue_scan.tsv']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))
    print(line, flush=True)

log('start, project=%s' % PROJECT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]
if len(human) != 459 or len(chicken) != 473:
    raise SystemExit('FATAL sequence lengths %d and %d, expected 459 and 473'
                     % (len(human), len(chicken)))
CHICKEN_SHA = hashlib.sha256(chicken.encode()).hexdigest()
EXPECT_SHA = '5d68e7c91dc3af3cd898a3f459617f3b22670f3dbdf9f798e8d740d8669a8aea'
if CHICKEN_SHA != EXPECT_SHA:
    raise SystemExit('FATAL the chicken sequence differs from the one the earlier steps '
                     'scored: %s against %s' % (CHICKEN_SHA, EXPECT_SHA))
log('sequences ok, chicken_sha256 matches the earlier steps')

AA = 'ACDEFGHIKLMNPQRSTVWY'
THRESHOLD = 1e-3
TOL = 5e-2

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pcol = {k: i for i, k in enumerate(rows[0])}
PSET = {}
for r in rows[1:]:
    cp, hp = int(r[pcol['chicken_pos']]), int(r[pcol['human_pos']])
    if chicken[cp - 1] != r[pcol['chicken_aa']] or human[hp - 1] != r[pcol['human_aa']]:
        raise SystemExit('FATAL the position set disagrees with the sequences at chicken '
                         '%d' % cp)
    PSET[cp] = {'chicken_aa': r[pcol['chicken_aa']], 'human_aa': r[pcol['human_aa']],
                'human_pos': hp, 'differs': r[pcol['differs']] == '1'}
PRIM = sorted(p for p, v in PSET.items() if v['differs'])
if len(PRIM) != 20:
    raise SystemExit('FATAL %d primaries, expected 20' % len(PRIM))

adm = json.load(open(os.path.join(PROJECT, 'data', 'admissible_construct.json')))
if sorted(adm['primaries']) != PRIM:
    raise SystemExit('FATAL the admissible construct lists different primaries from the '
                     'position set')
RESC = {int(k): v for k, v in adm['rescuer_positions'].items()}
for q, y in RESC.items():
    if q in PSET:
        raise SystemExit('FATAL rescuer position %d is in the pharmacophore set, so the '
                         'construct is not the admissible one' % q)
    if y == chicken[q - 1]:
        raise SystemExit('FATAL the rescuer at %d is not a substitution' % q)
log('admissible construct: %d primaries, %d rescuer positions %s'
    % (len(PRIM), len(RESC), sorted(RESC)))

con = json.load(open(os.path.join(PROJECT, 'data', 'construct_report.json')))
PROF5 = {x['chicken_pos']: x for x in con['part_b']['profile']}
rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'primary_substitution_risk.tsv'))]
rcol = {k: i for i, k in enumerate(rows[0])}
WT_BASE = {int(r[rcol['chicken_pos']]): float(r[rcol['p_human_wt_background']])
           for r in rows[1:]}


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
SCHEMA = 'v7-admissible-and-322'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()
AAID = {y: tok.convert_tokens_to_ids(y) for y in AA}
if any(v is None or v < 0 for v in AAID.values()):
    raise SystemExit('FATAL the tokenizer has no id for one of the amino acids')
offset = 1
NTOK = len(chicken) + 2


def build(primaries_in, rescuers_in, leave_out=None):
    s = list(chicken)
    if primaries_in:
        for p in PRIM:
            if p == leave_out:
                continue
            s[p - 1] = PSET[p]['human_aa']
    if rescuers_in:
        for q, y in RESC.items():
            if q == leave_out:
                continue
            s[q - 1] = y
    return ''.join(s)


BACKGROUNDS = [('wt', False, False), ('primaries', True, False),
               ('admissible', True, True)]
for name, pi, ri in BACKGROUNDS:
    s = build(pi, ri)
    log('background %-12s differs from chicken at %d positions'
        % (name, sum(1 for a, b in zip(s, chicken) if a != b)))

jobsA = [(name, t) for name, pi, ri in BACKGROUNDS for t in PRIM]
CKA = os.path.join(OUT, 'profileA_partial.tsv')
HA = chr(9).join(['schema_' + SCHEMA, 'background', 'chicken_pos', 'p_human'])
doneA = set()
if os.path.exists(CKA) and os.path.getsize(CKA) > 0:
    if open(CKA).readline().rstrip(chr(10)) != HA:
        raise SystemExit('FATAL checkpoint %s was written by a different schema; delete '
                         'it and rerun' % CKA)
    for line in open(CKA):
        p = line.rstrip(chr(10)).split(chr(9))
        if p[0] == 'row':
            doneA.add((p[1], int(p[2])))
todoA = [j for j in jobsA if j not in doneA]
log('part A: %d of %d to score' % (len(todoA), len(jobsA)))
BG = {name: (pi, ri) for name, pi, ri in BACKGROUNDS}
with open(CKA, 'a') as fh:
    if not doneA:
        fh.write(HA + chr(10))
        fh.flush()
    for b in range(0, len(todoA), BATCH):
        chunk = todoA[b:b + BATCH]
        seqs = [build(BG[n][0], BG[n][1], t) for (n, t) in chunk]
        enc = tok(seqs, return_tensors='pt')['input_ids']
        if enc.shape[1] != NTOK:
            raise SystemExit('FATAL a sequence tokenizes to %d tokens, expected %d'
                             % (enc.shape[1], NTOK))
        with torch.no_grad():
            logits = model(enc.cuda()).logits
        probs = torch.softmax(logits.float(), dim=-1)
        for k, (n, t) in enumerate(chunk):
            idx = t - 1 + offset
            if int(enc[k, idx]) != AAID[chicken[t - 1]]:
                raise SystemExit('FATAL the left-out position %d does not carry the '
                                 'chicken residue in background %s' % (t, n))
            fh.write(chr(9).join(['row', n, str(t),
                                  '%.6e' % float(probs[k, idx,
                                                       AAID[PSET[t]['human_aa']]])])
                     + chr(10))
        fh.flush()
log('part A scoring complete')


In [ ]:
PA = {}
for line in open(CKA):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        PA[(p[1], int(p[2]))] = float(p[3])
if len(PA) != len(jobsA):
    raise SystemExit('FATAL part A holds %d rows, %d expected' % (len(PA), len(jobsA)))

worst_wt = (0.0, None)
worst_pr = (0.0, None)
for t in PRIM:
    a, b = PA[('wt', t)], WT_BASE[t]
    rel = abs(a - b) / max(a, b, 1e-30)
    if rel > worst_wt[0]:
        worst_wt = (rel, (t, a, b))
    a, b = PA[('primaries', t)], PROF5[t]['primaries']
    rel = abs(a - b) / max(a, b, 1e-30)
    if rel > worst_pr[0]:
        worst_pr = (rel, (t, a, b))
if worst_wt[0] > TOL:
    t, a, b = worst_wt[1]
    raise SystemExit('FATAL the wild-type column does not reproduce step 1 at chicken %d, '
                     '%.6e against %.6e' % (t, a, b))
if worst_pr[0] > TOL:
    t, a, b = worst_pr[1]
    raise SystemExit('FATAL the primaries column does not reproduce step 5 at chicken %d, '
                     '%.6e against %.6e' % (t, a, b))
log('part A scale checks: wild type against step 1 worst %.2e, primaries against step 5 '
    'worst %.2e' % (worst_wt[0], worst_pr[0]))

colsA = ['chicken_pos', 'human_aa', 'p_wt', 'p_primaries', 'p_admissible',
         'p_step5_final', 'fold_admissible_over_wt', 'fold_admissible_over_step5']
rowsA = []
for t in PRIM:
    rowsA.append({'chicken_pos': t, 'human_aa': PSET[t]['human_aa'],
                  'p_wt': PA[('wt', t)], 'p_primaries': PA[('primaries', t)],
                  'p_admissible': PA[('admissible', t)],
                  'p_step5_final': PROF5[t]['primaries_rescuers_primary_wins'],
                  'fold_admissible_over_wt': PA[('admissible', t)] / PA[('wt', t)],
                  'fold_admissible_over_step5':
                      PA[('admissible', t)]
                      / PROF5[t]['primaries_rescuers_primary_wins']})
with open(os.path.join(OUT, 'admissible_profile.tsv'), 'w') as fh:
    fh.write(chr(9).join(colsA) + chr(10))
    for r in rowsA:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in colsA) + chr(10))

log('')
log('part A, every primary with the other nineteen and all six rescuers in place')
log('chicken  human   wild type   primaries   admissible   step 5 final  fold over wt')
for r in rowsA:
    log('%7d %6s %11.3e %11.3e %12.3e %14.3e %13.2f'
        % (r['chicken_pos'], r['human_aa'], r['p_wt'], r['p_primaries'],
           r['p_admissible'], r['p_step5_final'], r['fold_admissible_over_wt']))
belowA = [r['chicken_pos'] for r in rowsA if r['p_admissible'] < THRESHOLD]
below5 = [r['chicken_pos'] for r in rowsA if r['p_step5_final'] < THRESHOLD]
log('')
log('admissible construct below %.0e at %d of %d primaries%s'
    % (THRESHOLD, len(belowA), len(rowsA), (': %s' % belowA) if belowA else ''))
log('step 5 construct  below %.0e at %d of %d primaries%s'
    % (THRESHOLD, len(below5), len(rowsA), (': %s' % below5) if below5 else ''))
for t in sorted(int(k) for k in adm['chosen']):
    r = [x for x in rowsA if x['chicken_pos'] == t][0]
    log('  rescue at chicken %d: %s, chosen at %.6e, here %.6e, %s'
        % (t, adm['chosen'][str(t)]['label'], adm['chosen'][str(t)]['p'],
           r['p_admissible'],
           'holds' if r['p_admissible'] >= THRESHOLD else 'does not hold'))


In [ ]:
TARGET_B = 322
if TARGET_B not in PRIM:
    raise SystemExit('FATAL chicken %d is not a primary substitution' % TARGET_B)
cg = json.load(open(os.path.join(PROJECT, 'data', 'coupling_guided_rescue.json')))
st = json.load(open(os.path.join(PROJECT, 'data', 'structural_candidates.json')))
if str(TARGET_B) not in cg['targets'] or str(TARGET_B) not in st['targets']:
    raise SystemExit('FATAL chicken %d is missing from one of the two candidate rankings'
                     % TARGET_B)
COUP = [int(r['chicken_pos']) for r in cg['targets'][str(TARGET_B)]['top_coupled']]
NEAR = [int(c['chicken_pos']) for c in st['targets'][str(TARGET_B)]['candidates']]
CAND_B = sorted(set(COUP) | set(NEAR))
if TARGET_B in CAND_B:
    raise SystemExit('FATAL the target is in its own candidate list')
log('part B target chicken %d to human %s at human %d'
    % (TARGET_B, PSET[TARGET_B]['human_aa'], PSET[TARGET_B]['human_pos']))
log('  coupled top %d  : %s' % (len(COUP), COUP))
log('  nearest top %d  : %s' % (len(NEAR), NEAR))
log('  union %d        : %s' % (len(CAND_B), CAND_B))
log('  of those, %d lie inside the pharmacophore set: %s'
    % (len([p for p in CAND_B if p in PSET]), [p for p in CAND_B if p in PSET]))

BGB = build(True, False, TARGET_B)
if BGB[TARGET_B - 1] != chicken[TARGET_B - 1]:
    raise SystemExit('FATAL the part B background does not leave chicken %d alone'
                     % TARGET_B)
for p in PRIM:
    if p == TARGET_B:
        continue
    if BGB[p - 1] != PSET[p]['human_aa']:
        raise SystemExit('FATAL the part B background is not humanised at chicken %d' % p)
BASE_B = PROF5[TARGET_B]['primaries']
log('  background      : the %d other primaries written in, baseline %.6e from step 5'
    % (len(PRIM) - 1, BASE_B))

pairs_B = [(CAND_B[a], CAND_B[b]) for a in range(len(CAND_B))
           for b in range(a + 1, len(CAND_B))]
doubles_B = [(i, x, j, y) for (i, j) in pairs_B
             for x in AA if x != BGB[i - 1]
             for y in AA if y != BGB[j - 1]]
singles_B = [(j, y) for j in CAND_B for y in AA if y != BGB[j - 1]]
log('  pairs %d, doubles %d, singles %d'
    % (len(pairs_B), len(doubles_B), len(singles_B)))

BGB_IDS = tok(BGB, return_tensors='pt')['input_ids'][0]
READ_IDX_B = TARGET_B - 1 + offset
READ_TOK_B = AAID[PSET[TARGET_B]['human_aa']]
if int(BGB_IDS[READ_IDX_B]) != AAID[chicken[TARGET_B - 1]]:
    raise SystemExit('FATAL the read index does not carry the chicken residue')


def scoreB(variants, ckpt, header, key_of, apply_of):
    done = set()
    if os.path.exists(ckpt) and os.path.getsize(ckpt) > 0:
        if open(ckpt).readline().rstrip(chr(10)) != header:
            raise SystemExit('FATAL checkpoint %s was written by a different schema; '
                             'delete it and rerun' % ckpt)
        for line in open(ckpt):
            p = line.rstrip(chr(10)).split(chr(9))
            if p[0] == 'row':
                done.add(key_of(p))
        log('resuming %s, already scored %d' % (os.path.basename(ckpt), len(done)))
    todo = [v for v in variants if key_of(['row'] + [str(x) for x in v]) not in done]
    log('%s: %d to score of %d' % (os.path.basename(ckpt), len(todo), len(variants)))
    t0 = time.time()
    with open(ckpt, 'a') as fh:
        if not done:
            fh.write(header + chr(10))
            fh.flush()
        for b in range(0, len(todo), BATCH):
            chunk = todo[b:b + BATCH]
            batch = BGB_IDS.repeat(len(chunk), 1)
            for k, v in enumerate(chunk):
                apply_of(batch, k, v)
            with torch.no_grad():
                logits = model(batch.cuda()).logits
            sel = torch.softmax(logits[:, READ_IDX_B, :].float(), dim=-1)
            for k, v in enumerate(chunk):
                fh.write(chr(9).join(['row'] + [str(x) for x in v]
                                     + ['%.6e' % float(sel[k, READ_TOK_B])]) + chr(10))
            fh.flush()
            if (b // BATCH) % 50 == 0:
                el = (time.time() - t0) / 60
                log('  %d of %d, %.1f min elapsed, about %.0f min left'
                    % (b + len(chunk), len(todo), el,
                       (len(todo) - b - len(chunk)) / max((b + len(chunk)) / max(el, 1e-9),
                                                          1e-9)))
    log('%s complete, %.1f min' % (os.path.basename(ckpt), (time.time() - t0) / 60))


def applyB_single(batch, k, v):
    j, y = v
    batch[k, j - 1 + offset] = AAID[y]


def applyB_double(batch, k, v):
    i, x, j, y = v
    batch[k, i - 1 + offset] = AAID[x]
    batch[k, j - 1 + offset] = AAID[y]


SB = os.path.join(OUT, 'b322_singles_partial.tsv')
DB = os.path.join(OUT, 'b322_doubles_partial.tsv')
HSB = chr(9).join(['schema_' + SCHEMA, 'pos', 'aa', 'p_target'])
HDB = chr(9).join(['schema_' + SCHEMA, 'pos_i', 'aa_i', 'pos_j', 'aa_j', 'p_target'])
scoreB(singles_B, SB, HSB, lambda p: (int(p[1]), p[2]), applyB_single)
scoreB(doubles_B, DB, HDB, lambda p: (int(p[1]), p[2], int(p[3]), p[4]), applyB_double)


In [ ]:
singB = {}
for line in open(SB):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        singB[(int(p[1]), p[2])] = float(p[3])
dblB = {}
for line in open(DB):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        dblB[(int(p[1]), p[2], int(p[3]), p[4])] = float(p[5])
if len(singB) != len(singles_B) or len(dblB) != len(doubles_B):
    raise SystemExit('FATAL part B holds %d singles and %d doubles, %d and %d expected'
                     % (len(singB), len(dblB), len(singles_B), len(doubles_B)))

PREV = {}
scan = open(os.path.join(PROJECT, 'data', 'esm2_step2', 'rescue_scan.tsv'))
head = scan.readline().rstrip(chr(10)).split(chr(9))
scol = {k: i for i, k in enumerate(head)}
for line in scan:
    r = line.rstrip(chr(10)).split(chr(9))
    if int(r[scol['target_pos']]) != TARGET_B:
        continue
    PREV[(int(r[scol['rescuer_pos']]), r[scol['rescuer_to']])] = float(r[scol['p_target']])

colsB = ['pos_i', 'from_construct_i', 'from_chicken_i', 'to_i',
         'pos_j', 'from_construct_j', 'from_chicken_j', 'to_j',
         'p_target', 'p_single_i', 'p_single_j', 'best_single_of_pair',
         'fold_vs_baseline', 'fold_vs_best_single_of_pair', 'crosses',
         'in_set_i', 'in_set_j', 'admissible']
rowsB = []
for (i, x, j, y), p in dblB.items():
    pi, pj = singB[(i, x)], singB[(j, y)]
    rowsB.append({'pos_i': i, 'from_construct_i': BGB[i - 1],
                  'from_chicken_i': chicken[i - 1], 'to_i': x,
                  'pos_j': j, 'from_construct_j': BGB[j - 1],
                  'from_chicken_j': chicken[j - 1], 'to_j': y,
                  'p_target': p, 'p_single_i': pi, 'p_single_j': pj,
                  'best_single_of_pair': max(pi, pj),
                  'fold_vs_baseline': p / BASE_B,
                  'fold_vs_best_single_of_pair': p / max(pi, pj),
                  'crosses': p >= THRESHOLD,
                  'in_set_i': i in PSET, 'in_set_j': j in PSET,
                  'admissible': (i not in PSET) and (j not in PSET)})
rowsB.sort(key=lambda r: -r['p_target'])
with open(os.path.join(OUT, 'b322_double_rescue.tsv'), 'w') as fh:
    fh.write(chr(9).join(colsB) + chr(10))
    for r in rowsB:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in colsB) + chr(10))
with open(os.path.join(OUT, 'b322_singles.tsv'), 'w') as fh:
    fh.write(chr(9).join(['pos', 'from_construct', 'from_chicken', 'to',
                          'p_construct', 'p_wt_step2']) + chr(10))
    for (j, y), v in sorted(singB.items()):
        pv = PREV.get((j, y))
        fh.write(chr(9).join([str(j), BGB[j - 1], chicken[j - 1], y, '%.6e' % v,
                              ('%.6e' % pv) if pv else '']) + chr(10))

crossB = [r for r in rowsB if r['crosses']]
adm_cross = [r for r in crossB if r['admissible']]
best_single_B = max(singB.items(), key=lambda kv: kv[1])
folds = sorted(r['fold_vs_best_single_of_pair'] for r in rowsB)
log('')
log('part B, chicken %d by pairs in the construct background' % TARGET_B)
log('  baseline        : %.6e' % BASE_B)
log('  singles         : %d, strongest %s%d at %.6e'
    % (len(singB), BGB[best_single_B[0][0] - 1], best_single_B[0][0], best_single_B[1]))
log('  doubles         : %d over %d pairs, %d crossing %.0e, %d of those admissible'
    % (len(rowsB), len(pairs_B), len(crossB), THRESHOLD, len(adm_cross)))
log('  fold over the better half: median %.2f, max %.2f'
    % (folds[len(folds) // 2], max(folds)))
log('  the five strongest')
for r in rowsB[:5]:
    log('    %s%d%s with %s%d%s at %.6e, %.1f-fold over baseline, %.2f-fold over the '
        'better half, admissible %s'
        % (r['from_construct_i'], r['pos_i'], r['to_i'],
           r['from_construct_j'], r['pos_j'], r['to_j'], r['p_target'],
           r['fold_vs_baseline'], r['fold_vs_best_single_of_pair'], r['admissible']))
if adm_cross:
    r = adm_cross[0]
    log('  strongest admissible crossing pair: %s%d%s with %s%d%s at %.6e'
        % (r['from_construct_i'], r['pos_i'], r['to_i'],
           r['from_construct_j'], r['pos_j'], r['to_j'], r['p_target']))
else:
    log('  no crossing pair leaves the pharmacophore set untouched')
log('  verdict         : %s'
    % ('chicken %d is rescued by a pair in the construct background' % TARGET_B
       if crossB else
       'no pair at these %d positions lifts chicken %d above %.0e in the construct '
       'background' % (len(CAND_B), TARGET_B, THRESHOLD)))

summary = {'schema': SCHEMA, 'model': MODEL, 'chicken_sha256': CHICKEN_SHA,
           'threshold': THRESHOLD,
           'part_a': {'backgrounds': [b[0] for b in BACKGROUNDS],
                      'rescuer_positions': {str(k): v for k, v in RESC.items()},
                      'wt_vs_step1_worst': worst_wt[0],
                      'primaries_vs_step5_worst': worst_pr[0],
                      'below_threshold_admissible': belowA,
                      'below_threshold_step5': below5,
                      'profile': rowsA},
           'part_b': {'target': TARGET_B, 'coupled': COUP, 'nearest': NEAR,
                      'candidates': CAND_B, 'baseline': BASE_B,
                      'background': 'the other %d primaries written in' % (len(PRIM) - 1),
                      'n_pairs': len(pairs_B), 'n_doubles': len(rowsB),
                      'n_singles': len(singB),
                      'best_single': {'pos': best_single_B[0][0],
                                      'to': best_single_B[0][1], 'p': best_single_B[1]},
                      'n_crossing': len(crossB),
                      'n_crossing_admissible': len(adm_cross),
                      'best_double': rowsB[0],
                      'best_admissible_double': adm_cross[0] if adm_cross else None,
                      'top_doubles': rowsB[:20],
                      'fold_over_better_half': {'median': folds[len(folds) // 2],
                                                'max': max(folds)}}}
with open(os.path.join(OUT, 'step7_summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)
log('')
log('wrote admissible_profile.tsv, b322_double_rescue.tsv, b322_singles.tsv and '
    'step7_summary.json')
log('copy esm2_step7/ from Drive into data/ and run codes/21_step7_read.py')
